Task 2. Fine-tune one pretrained existing deep learning model, e.g. VGGish, and use the 
same training data and test data split to evaluate the model.  

In [1]:

import os, json, time, glob, numpy as np, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt, seaborn as sns

SEED = 42
keras.utils.set_random_seed(SEED)

DATA_DIR = '/Users/annacasanovaspoirier/Desktop/MSC/Machine Learning for Sound and Music/music-genre-classification-2/data/images_original'   # genre subfolders with PNGs

In [3]:
from pathlib import Path

DATA_DIR = Path('./data/images_original')    # carpeta con las subcarpetas de géneros (blues, classical, ...)
WAV_DIR  = Path('./data/genres_original')    # solo para VGGish
SPLIT_FILE = Path('./split_seed42.json')     # se guarda junto al notebook

In [10]:
genres = sorted(g for g in os.listdir(DATA_DIR) if (Path(DATA_DIR) / g).is_dir())
class_to_idx = {g: i for i, g in enumerate(genres)}
N_CLASSES = len(genres)
print(genres, N_CLASSES)          # must be 10 genres

def to_labels(ps):
    return np.array([class_to_idx[Path(p).parent.name] for p in ps], dtype=np.int32)

tr_y, va_y, te_y = to_labels(tr_p), to_labels(va_p), to_labels(te_p)
assert tr_y.min() == 0 and tr_y.max() == 9
print(np.bincount(tr_y), np.bincount(va_y), np.bincount(te_y))   # about 80 / 9 / 20 per genre

['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock'] 10
[72 72 72 72 72 71 72 72 72 72] [8 8 8 8 8 8 8 8 8 8] [20 20 20 20 20 20 20 20 20 20]


In [11]:
if SPLIT_FILE.exists():
    split = json.load(open(SPLIT_FILE))
    tr_p, va_p, te_p = map(np.array, (split['train'], split['val'], split['test']))
    lab = lambda ps: np.array([class_to_idx[Path(p).name.split('.')[0]] for p in ps])
    tr_y, va_y, te_y = lab(tr_p), lab(va_p), lab(te_p)
else:
    tr_p, te_p, tr_y, te_y = train_test_split(paths, labels, test_size=0.2,
                                              stratify=labels, random_state=SEED)
    tr_p, va_p, tr_y, va_y = train_test_split(tr_p, tr_y, test_size=0.1,
                                              stratify=tr_y, random_state=SEED)
    json.dump({'train': tr_p.tolist(), 'val': va_p.tolist(), 'test': te_p.tolist()},
              open(SPLIT_FILE, 'w'))
print(len(tr_p), len(va_p), len(te_p))

KeyError: 'classical00026'

In [6]:
print(tf.config.list_physical_devices('GPU'))

[]


In [8]:
print(tr_y[:5], tr_y.dtype)
print(va_y[:5], va_y.dtype)
print(tr_p[:2])

[2 7 6 8 3] int64
[ 7 10  8  6  4] int64
['/Users/annacasanovaspoirier/Desktop/MSC/Machine Learning for Sound and Music/music-genre-classification-2/data/images_original/classical/classical00026.png'
 '/Users/annacasanovaspoirier/Desktop/MSC/Machine Learning for Sound and Music/music-genre-classification-2/data/images_original/metal/metal00042.png']


In [12]:
x, y = next(iter(make_ds(tr_p, tr_y, BACKBONES['vgg16'][1], train=True)))
print(x.shape, x.dtype, y.shape, y.dtype)   # (32, 224, 224, 3) float32 (32,) int32

results = {}
results['vgg16'] = run_experiment('vgg16')

(32, 224, 224, 3) <dtype: 'float32'> (32,) <dtype: 'int32'>
Epoch 1/15


ValueError: dtype='string' is not a valid dtype for Keras type promotion.

In [7]:
IMG_SIZE = (224, 224)
BATCH = 32
N_CLASSES = len(genres)

BACKBONES = {
    'vgg16':     (keras.applications.VGG16,        keras.applications.vgg16.preprocess_input),
    'resnet50':  (keras.applications.ResNet50,     keras.applications.resnet50.preprocess_input),
    'effnetb0':  (keras.applications.EfficientNetB0, keras.applications.efficientnet.preprocess_input),
}

def make_ds(paths, labels, preprocess, train=False):
    def load(p, y):
        img = tf.io.read_file(p)
        img = tf.image.decode_png(img, channels=3)       # drops the alpha channel
        img = tf.image.resize(img, IMG_SIZE)
        return preprocess(img), y
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if train: ds = ds.shuffle(len(paths), seed=SEED)
    return ds.map(load, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH).prefetch(tf.data.AUTOTUNE)

def build_model(name, dropout=0.3):
    Base, _ = BACKBONES[name]
    base = Base(include_top=False, weights='imagenet', input_shape=IMG_SIZE + (3,))
    base.trainable = False
    inp = keras.Input(IMG_SIZE + (3,))
    x = base(inp, training=False)             # keeps BatchNorm in inference mode
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(N_CLASSES, activation='softmax')(x)
    return keras.Model(inp, out), base

def run_experiment(name, head_epochs=15, ft_epochs=20, unfreeze_last=30, lr_head=1e-3, lr_ft=1e-5):
    _, prep = BACKBONES[name]
    train_ds = make_ds(tr_p, tr_y, prep, train=True)
    val_ds   = make_ds(va_p, va_y, prep)
    test_ds  = make_ds(te_p, te_y, prep)

    model, base = build_model(name)
    cb = [keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor='val_accuracy')]

    t0 = time.time()
    # Phase 1: train only the new head
    model.compile(keras.optimizers.Adam(lr_head), 'sparse_categorical_crossentropy', ['accuracy'])
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=head_epochs, callbacks=cb, verbose=2)

    # Phase 2: unfreeze the top layers and fine-tune with a low learning rate
    base.trainable = True
    for l in base.layers[:-unfreeze_last]:
        l.trainable = False
    for l in base.layers:                      # keep BN frozen
        if isinstance(l, layers.BatchNormalization): l.trainable = False
    model.compile(keras.optimizers.Adam(lr_ft), 'sparse_categorical_crossentropy', ['accuracy'])
    h2 = model.fit(train_ds, validation_data=val_ds, epochs=ft_epochs, callbacks=cb, verbose=2)
    train_time = time.time() - t0

    # Evaluate on the test set ONLY after choosing the config
    t0 = time.time()
    probs = model.predict(test_ds, verbose=0)
    infer_time = (time.time() - t0) / len(te_p)
    preds = probs.argmax(1)
    acc = accuracy_score(te_y, preds)
    print(f'{name}: test acc={acc:.4f} | train {train_time:.0f}s | {infer_time*1000:.1f} ms/song | params {model.count_params():,}')
    return dict(model=model, probs=probs, preds=preds, acc=acc,
                train_time=train_time, infer_time=infer_time, params=model.count_params(),
                hist=(h1.history, h2.history))

results = {}
results['vgg16'] = run_experiment('vgg16')

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 13s 0us/step
Epoch 1/15


/Users/annacasanovaspoirier/miniconda3/envs/sms/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


ValueError: dtype='string' is not a valid dtype for Keras type promotion.

In [ ]:
def evaluate(res, title):
    print(classification_report(te_y, res['preds'], target_names=genres))
    cm = confusion_matrix(te_y, res['preds'])
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=genres, yticklabels=genres, cmap='Blues')
    plt.title(f"{title} (acc={res['acc']:.3f})"); plt.ylabel('True'); plt.xlabel('Pred'); plt.show()

evaluate(results['vgg16'], 'VGG16 fine-tuned')

## LAB 2 RESEARCH QUESTIONS


1. Do learned features beat hand-crafted descriptors? Compare all models on the 
same 200-song test set. You will need to rerun the Lab 1 models on the new split for a fair comparison.

2. Do CNNs and descriptor models make the same mistakes? Compare your confusion 
matrices and check how much their errors overlap. Is the genre-similarity structure found in Lab 1 the same here, or is it partly an effect of the descriptors? 

3. Are the two approaches complementary? Concatenate CNN embeddings with the 
Lab 1 descriptors, or ensemble the two models. If their errors don’t overlap much, the combination should improve accuracy. 

4. Is the extra cost worth it? Compare accuracy against training time, inference time and model size. A random forest on descriptors may come close to a CNN at a fraction of the cost. 

5. How much does one 80/20 split tell you? With 200 test songs, one song is 0.5% of accuracy. Repeat the split with several seeds, before claiming that one model beats another. 

6. How does data size affect the comparison? Plot learning curves using 10%, 25%, 
50% and 100% of the training set for the scratch CNN, and the best Lab 1 model. 
 
7. Should a CNN treat a spectrogram like a photo? Compare square 3×3 kernels with 
“musically motivated” shapes (as in Pons et al.).  

8. How do depth and parameter count impact? Plot test accuracy against the number 
of parameters (in different CNN configurations tried). Where does overfitting start? 

9. Would fine-tuning a pretrained model (e.g. VGG16, VGGish Clap) improve the 
model? Is a vision model or an audio model the better starting point?  

10. Frozen embeddings + traditional ML algorithm, or full fine-tuning?